<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/02_Data_Preprocessing_and_Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================================
# CELL 02.0 — AIR-LLM NOTEBOOK 02 INITIALIZATION
#             ROBUST GOOGLE DRIVE MOUNT + CONFIGURATION LOAD
# ============================================================

from pathlib import Path
import os
import json
import random
import warnings

import yaml
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("AIR-LLM — NOTEBOOK 02")
print("DATA PREPROCESSING AND EXPERIMENTAL SPLITTING")
print("=" * 100)

# ============================================================
# 1. GOOGLE DRIVE — USE A CLEAN MOUNTPOINT
# ============================================================

from google.colab import drive

# IMPORTANT:
# /content/drive is a stale/populated Colab directory in the
# current runtime. Therefore we intentionally use a different
# mountpoint.
#
# This DOES NOT change your Google Drive project.
#
# Google Drive:
#     MyDrive/AIR_LLM_Research
#
# Runtime mount:
#     /content/air_llm_drive

DRIVE_MOUNTPOINT = Path(
    "/content/air_llm_drive"
)

# ------------------------------------------------------------
# Remove only our temporary runtime mountpoint if necessary.
# We NEVER delete /content/drive and NEVER delete Drive data.
# ------------------------------------------------------------

if DRIVE_MOUNTPOINT.exists():

    try:

        # Attempt normal unmount first.
        drive.flush_and_unmount()

        print(
            "Existing temporary Drive mount released."
        )

    except Exception:

        pass

# ------------------------------------------------------------
# Create a clean mountpoint only if it does not exist.
# ------------------------------------------------------------

DRIVE_MOUNTPOINT.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Mount Google Drive at the clean mountpoint.
# ------------------------------------------------------------

try:

    drive.mount(
        str(DRIVE_MOUNTPOINT),
        force_remount=False
    )

except ValueError as e:

    # If this temporary mountpoint somehow became stale,
    # force-remount it.
    if "Mountpoint must not already contain files" in str(e):

        try:

            drive.flush_and_unmount()

        except Exception:

            pass

        # Recreate an empty runtime mountpoint.
        if DRIVE_MOUNTPOINT.exists():

            try:

                for item in DRIVE_MOUNTPOINT.iterdir():

                    if item.is_dir():

                        import shutil

                        shutil.rmtree(
                            item
                        )

                    else:

                        item.unlink()

            except Exception as cleanup_error:

                raise RuntimeError(
                    "\nUnable to clean the temporary "
                    "Colab mountpoint:\n"
                    f"{DRIVE_MOUNTPOINT}\n\n"
                    f"Cleanup error: {cleanup_error}"
                )

        drive.mount(
            str(DRIVE_MOUNTPOINT),
            force_remount=False
        )

    else:

        raise RuntimeError(
            "\nGoogle Drive could not be mounted at the "
            "temporary AIR-LLM mountpoint.\n\n"
            f"Mountpoint: {DRIVE_MOUNTPOINT}\n"
            f"Original error: {e}"
        )

print(
    f"\n✓ Google Drive mounted at:\n"
    f"  {DRIVE_MOUNTPOINT}"
)

# ============================================================
# 2. CANONICAL AIR-LLM PROJECT ROOT
# ============================================================

# This is the ONLY project location used by Notebook 02.
#
# Notice that this is still the same Google Drive project:
#
# MyDrive/AIR_LLM_Research
#
# We are simply accessing MyDrive through the new runtime
# mountpoint.

PROJECT_ROOT = (
    DRIVE_MOUNTPOINT
    / "MyDrive"
    / "AIR_LLM_Research"
)

print(
    "\nChecking AIR-LLM project root..."
)

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        "\nAIR-LLM project root was not found.\n\n"
        f"Expected:\n{PROJECT_ROOT}\n\n"
        "Your Notebook 00 project should exist at:\n"
        "MyDrive/AIR_LLM_Research"
    )

print(
    f"✓ AIR-LLM project found:\n"
    f"  {PROJECT_ROOT}"
)

# ============================================================
# 3. CONFIGURATION PATHS
# ============================================================

CONFIG_DIR = (
    PROJECT_ROOT
    / "config"
)

MASTER_CONFIG_PATH = (
    CONFIG_DIR
    / "config.yaml"
)

DATASET_REGISTRY_PATH = (
    CONFIG_DIR
    / "dataset_registry.yaml"
)

FEATURE_TARGET_REGISTRY_PATH = (
    CONFIG_DIR
    / "feature_target_registry.yaml"
)

SCENARIO_REGISTRY_PATH = (
    CONFIG_DIR
    / "experiment_scenario_registry.yaml"
)

EXPERIMENT_DESIGN_PATH = (
    CONFIG_DIR
    / "experiment_design.yaml"
)

ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
)

RESEARCH_MANIFEST_PATH = (
    ARTIFACTS_DIR
    / "research_manifest.json"
)

CONFIGURATION_LOCK_PATH = (
    ARTIFACTS_DIR
    / "configuration_lock.json"
)

# ============================================================
# 4. VERIFY NOTEBOOK 00 FILES
# ============================================================

REQUIRED_NOTEBOOK00_FILES = {

    "Master Config":
        MASTER_CONFIG_PATH,

    "Dataset Registry":
        DATASET_REGISTRY_PATH,

    "Feature/Target Registry":
        FEATURE_TARGET_REGISTRY_PATH,

    "Scenario Registry":
        SCENARIO_REGISTRY_PATH,

    "Experiment Design":
        EXPERIMENT_DESIGN_PATH,

    "Research Manifest":
        RESEARCH_MANIFEST_PATH,

    "Configuration Lock":
        CONFIGURATION_LOCK_PATH,
}

print(
    "\nChecking locked Notebook 00 configuration..."
)

MISSING_FILES = []

for name, path in REQUIRED_NOTEBOOK00_FILES.items():

    if not path.exists():

        MISSING_FILES.append(
            f"{name}: {path}"
        )

    elif path.stat().st_size == 0:

        MISSING_FILES.append(
            f"{name}: EMPTY FILE"
        )

    else:

        print(
            f"  ✓ {name:<28} "
            f"{path.stat().st_size:,} bytes"
        )

# ============================================================
# 5. STOP IF NOTEBOOK 00 IS NOT AVAILABLE
# ============================================================

if MISSING_FILES:

    print(
        "\n" + "=" * 100
    )

    print(
        "NOTEBOOK 00 CONFIGURATION NOT FOUND"
    )

    print(
        "=" * 100
    )

    for item in MISSING_FILES:

        print(
            f"  ✗ {item}"
        )

    raise FileNotFoundError(
        "\nNotebook 00 configuration cannot be accessed "
        "from the current Google Drive mount.\n\n"
        "Do NOT recreate configuration files in Notebook 02."
    )

# ============================================================
# 6. LOAD MASTER CONFIGURATION
# ============================================================

with open(
    MASTER_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    MASTER_CONFIG = yaml.safe_load(
        file
    )

# ============================================================
# 7. LOAD DATASET REGISTRY
# ============================================================

with open(
    DATASET_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    DATASET_REGISTRY = yaml.safe_load(
        file
    )

# ============================================================
# 8. LOAD FEATURE/TARGET REGISTRY
# ============================================================

with open(
    FEATURE_TARGET_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    FEATURE_TARGET_REGISTRY = yaml.safe_load(
        file
    )

# ============================================================
# 9. LOAD EXPERIMENT SCENARIO REGISTRY
# ============================================================

with open(
    SCENARIO_REGISTRY_PATH,
    "r",
    encoding="utf-8"
) as file:

    EXPERIMENT_SCENARIO_REGISTRY = yaml.safe_load(
        file
    )

# ============================================================
# 10. LOAD EXPERIMENT DESIGN
# ============================================================

with open(
    EXPERIMENT_DESIGN_PATH,
    "r",
    encoding="utf-8"
) as file:

    EXPERIMENT_DESIGN = yaml.safe_load(
        file
    )

# ============================================================
# 11. LOAD RESEARCH MANIFEST
# ============================================================

with open(
    RESEARCH_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as file:

    RESEARCH_MANIFEST = json.load(
        file
    )

# ============================================================
# 12. LOAD CONFIGURATION LOCK
# ============================================================

with open(
    CONFIGURATION_LOCK_PATH,
    "r",
    encoding="utf-8"
) as file:

    CONFIGURATION_LOCK = json.load(
        file
    )

# ============================================================
# 13. REPRODUCIBILITY SEED
# ============================================================

SEED = 42

try:

    SEED = int(
        MASTER_CONFIG
        .get(
            "reproducibility",
            {}
        )
        .get(
            "seed",
            42
        )
    )

except Exception:

    SEED = 42

random.seed(
    SEED
)

np.random.seed(
    SEED
)

# ============================================================
# 14. NOTEBOOK 02 DIRECTORIES
# ============================================================

DATA_PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

DATA_REFERENCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "reference"
)

EXPERIMENT_DIR = (
    PROJECT_ROOT
    / "experiments"
)

PREPROCESSING_DIR = (
    EXPERIMENT_DIR
    / "preprocessing"
)

NOTEBOOK02_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "processed"
    / "notebook_02"
)

for directory in [

    PREPROCESSING_DIR,

    NOTEBOOK02_RESULTS_DIR,

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ============================================================
# 15. FINAL CONFIGURATION HANDSHAKE
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 00 → NOTEBOOK 02 CONFIGURATION HANDSHAKE"
)

print(
    "=" * 100
)

print(
    f"Google Drive Mount : {DRIVE_MOUNTPOINT}"
)

print(
    f"Project Root       : {PROJECT_ROOT}"
)

print(
    f"Master Config      : {MASTER_CONFIG_PATH}"
)

print(
    f"Dataset Registry   : {DATASET_REGISTRY_PATH}"
)

print(
    f"Feature/Target Reg.: {FEATURE_TARGET_REGISTRY_PATH}"
)

print(
    f"Scenario Registry  : {SCENARIO_REGISTRY_PATH}"
)

print(
    f"Experiment Design  : {EXPERIMENT_DESIGN_PATH}"
)

print(
    f"Random Seed        : {SEED}"
)

print(
    "\nLocked Notebook 00 artifacts:"
)

for name in REQUIRED_NOTEBOOK00_FILES:

    print(
        f"  ✓ {name}"
    )

print(
    "\n" + "=" * 100
)

print(
    "NOTEBOOK 02 INITIALIZATION PASSED"
)

print(
    "LOCKED NOTEBOOK 00 CONFIGURATION LOADED SUCCESSFULLY"
)

print(
    "READY FOR 02.1 LOAD CANONICAL DATASETS"
)

print(
    "=" * 100
)

AIR-LLM — NOTEBOOK 02
DATA PREPROCESSING AND EXPERIMENTAL SPLITTING
Mounted at /content/air_llm_drive

✓ Google Drive mounted at:
  /content/air_llm_drive

Checking AIR-LLM project root...
✓ AIR-LLM project found:
  /content/air_llm_drive/MyDrive/AIR_LLM_Research

Checking locked Notebook 00 configuration...
  ✓ Master Config                9,455 bytes
  ✓ Dataset Registry             656 bytes
  ✓ Feature/Target Registry      543 bytes
  ✓ Scenario Registry            1,245 bytes
  ✓ Experiment Design            310 bytes
  ✓ Research Manifest            1,629 bytes
  ✓ Configuration Lock           638 bytes

NOTEBOOK 00 → NOTEBOOK 02 CONFIGURATION HANDSHAKE
Google Drive Mount : /content/air_llm_drive
Project Root       : /content/air_llm_drive/MyDrive/AIR_LLM_Research
Master Config      : /content/air_llm_drive/MyDrive/AIR_LLM_Research/config/config.yaml
Dataset Registry   : /content/air_llm_drive/MyDrive/AIR_LLM_Research/config/dataset_registry.yaml
Feature/Target Reg.: /content/air

In [6]:
# ============================================================
# CELL 02.1 — LOAD CANONICAL DATASETS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("02.1 — LOAD CANONICAL DATASETS")
print("=" * 100)

# ============================================================
# 1. VERIFY NOTEBOOK 02 INITIALIZATION
# ============================================================

if "PROJECT_ROOT" not in globals():
    raise RuntimeError(
        "PROJECT_ROOT is not defined.\n"
        "Run CELL 02.0 first."
    )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"AIR-LLM project root not found:\n{PROJECT_ROOT}"
    )

# ============================================================
# 2. DEFINE CANONICAL DATA LOCATION
# ============================================================

DATA_PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

if not DATA_PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed-data directory not found:\n"
        f"{DATA_PROCESSED_DIR}\n\n"
        "Notebook 01 must be completed successfully first."
    )

# ============================================================
# 3. FIXED AIR-LLM DATASET REGISTRY
# ============================================================

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

DATASET_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

# ============================================================
# 4. LOCATE CANONICAL DATASETS
# ============================================================

CANONICAL_FILE_MAP = {}

for dataset_id in DATASET_IDS:

    candidate_paths = [

        DATA_PROCESSED_DIR
        / f"{dataset_id}_canonical.csv",

        DATA_PROCESSED_DIR
        / f"{dataset_id}.csv"

    ]

    existing_paths = [
        path
        for path in candidate_paths
        if path.exists()
        and path.is_file()
        and path.stat().st_size > 0
    ]

    if not existing_paths:

        raise FileNotFoundError(
            f"\nCanonical dataset not found for: "
            f"{dataset_id}\n\n"
            f"Expected one of:\n"
            + "\n".join(
                f"  {path}"
                for path in candidate_paths
            )
        )

    CANONICAL_FILE_MAP[
        dataset_id
    ] = existing_paths[0]

# ============================================================
# 5. LOAD CANONICAL DATASETS
# ============================================================

CANONICAL_DATASETS = {}

for dataset_id in DATASET_IDS:

    file_path = CANONICAL_FILE_MAP[
        dataset_id
    ]

    df = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        low_memory=False
    )

    # --------------------------------------------------------
    # Basic structural validation
    # --------------------------------------------------------

    if df.empty:

        raise RuntimeError(
            f"Canonical dataset is empty: "
            f"{dataset_id}"
        )

    # --------------------------------------------------------
    # Normalize accidental unnamed index column
    # --------------------------------------------------------

    unnamed_columns = [
        column
        for column in df.columns
        if str(column).lower().startswith(
            "unnamed:"
        )
    ]

    if unnamed_columns:

        df = df.drop(
            columns=unnamed_columns
        )

    # --------------------------------------------------------
    # Validate target
    # --------------------------------------------------------

    target = DATASET_TARGETS[
        dataset_id
    ]

    if target not in df.columns:

        raise RuntimeError(
            f"\nTarget column '{target}' "
            f"not found in {dataset_id}.\n\n"
            f"Available columns:\n"
            f"{df.columns.tolist()}"
        )

    # --------------------------------------------------------
    # Store dataset
    # --------------------------------------------------------

    CANONICAL_DATASETS[
        dataset_id
    ] = df

# ============================================================
# 6. DATASET SUMMARY
# ============================================================

LOAD_SUMMARY = pd.DataFrame(
    [
        {
            "dataset_id": dataset_id,
            "file": str(
                CANONICAL_FILE_MAP[
                    dataset_id
                ]
            ),
            "rows": int(
                df.shape[0]
            ),
            "columns": int(
                df.shape[1]
            ),
            "target": DATASET_TARGETS[
                dataset_id
            ],
            "target_exists":
                DATASET_TARGETS[
                    dataset_id
                ] in df.columns,
            "missing_cells": int(
                df.isna().sum().sum()
            ),
            "duplicate_rows": int(
                df.duplicated().sum()
            )
        }

        for dataset_id, df
        in CANONICAL_DATASETS.items()
    ]
)

display(
    LOAD_SUMMARY
)

# ============================================================
# 7. VERIFY ALL DATASETS ARE AVAILABLE FOR 02.2
# ============================================================

if set(CANONICAL_DATASETS.keys()) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Canonical dataset loading is incomplete.\n"
        f"Expected: {DATASET_IDS}\n"
        f"Loaded: {list(CANONICAL_DATASETS.keys())}"
    )

# ============================================================
# 8. PERSIST LOAD SUMMARY
# ============================================================

NOTEBOOK02_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "processed"
    / "notebook_02"
)

NOTEBOOK02_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOAD_SUMMARY_PATH = (
    NOTEBOOK02_RESULTS_DIR
    / "canonical_dataset_load_summary.csv"
)

LOAD_SUMMARY.to_csv(
    LOAD_SUMMARY_PATH,
    index=False
)

# ============================================================
# 9. FINAL STATUS
# ============================================================

print()
print("=" * 100)
print("02.1 CANONICAL DATASET LOADING COMPLETE")
print("=" * 100)

for dataset_id, df in CANONICAL_DATASETS.items():

    print(
        f"✓ {dataset_id:<20} "
        f"rows={df.shape[0]:>8,}  "
        f"columns={df.shape[1]:>3}  "
        f"target={DATASET_TARGETS[dataset_id]}"
    )

print()
print(
    f"Summary saved:\n{LOAD_SUMMARY_PATH}"
)

print()
print(
    "CANONICAL_DATASETS is now available for CELL 02.2."
)

print("=" * 100)

02.1 — LOAD CANONICAL DATASETS


,dataset_id,file,rows,columns,target,target_exists,missing_cells,duplicate_rows
0,adult_income,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,32561,15,income,True,4262,24
1,bank_marketing,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,45211,17,y,True,0,0
2,diabetes_130us,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,101766,48,readmitted,True,374017,0



02.1 CANONICAL DATASET LOADING COMPLETE
✓ adult_income         rows=  32,561  columns= 15  target=income
✓ bank_marketing       rows=  45,211  columns= 17  target=y
✓ diabetes_130us       rows= 101,766  columns= 48  target=readmitted

Summary saved:
/content/air_llm_drive/MyDrive/AIR_LLM_Research/results/processed/notebook_02/canonical_dataset_load_summary.csv

CANONICAL_DATASETS is now available for CELL 02.2.


In [7]:
# ============================================================
# CELL 02.2 — DATA-TYPE IDENTIFICATION
# ============================================================

DTYPE_RECORDS = []

DATASET_FEATURE_TYPES = {}

for dataset_id, df in CANONICAL_DATASETS.items():

    target = DATASET_TARGETS[
        dataset_id
    ]

    feature_columns = [
        column
        for column in df.columns
        if column != target
    ]

    numerical_columns = (
        df[feature_columns]
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        column
        for column in feature_columns
        if column not in numerical_columns
    ]


    # --------------------------------------------------------
    # Target type
    # --------------------------------------------------------

    target_is_numeric = (
        pd.api.types.is_numeric_dtype(
            df[target]
        )
    )


    DATASET_FEATURE_TYPES[
        dataset_id
    ] = {
        "target": target,
        "features": feature_columns,
        "numerical": numerical_columns,
        "categorical": categorical_columns,
        "target_numeric": target_is_numeric
    }


    for column in df.columns:

        if column == target:

            variable_role = "target"

        elif column in numerical_columns:

            variable_role = "numerical_feature"

        else:

            variable_role = "categorical_feature"


        DTYPE_RECORDS.append(
            {
                "dataset_id":
                    dataset_id,

                "column":
                    column,

                "pandas_dtype":
                    str(df[column].dtype),

                "variable_role":
                    variable_role,

                "unique_values":
                    int(
                        df[column]
                        .nunique(
                            dropna=True
                        )
                    )
            }
        )


DTYPE_DF = pd.DataFrame(
    DTYPE_RECORDS
)

display(DTYPE_DF)

print()
print("Data-type identification completed.")

,dataset_id,column,pandas_dtype,variable_role,unique_values
0,adult_income,age,int64,numerical_feature,73
1,adult_income,workclass,object,categorical_feature,8
2,adult_income,fnlwgt,int64,numerical_feature,21648
3,adult_income,education,object,categorical_feature,16
4,adult_income,education_num,int64,numerical_feature,16
...,...,...,...,...,...
75,diabetes_130us,metformin_rosiglitazone,object,categorical_feature,2
76,diabetes_130us,metformin_pioglitazone,object,categorical_feature,2
77,diabetes_130us,change,object,categorical_feature,2
78,diabetes_130us,diabetesMed,object,categorical_feature,2



Data-type identification completed.


In [8]:
# ============================================================
# CELL 02.2 — DATA-TYPE IDENTIFICATION
# ============================================================

DTYPE_RECORDS = []

DATASET_FEATURE_TYPES = {}

for dataset_id, df in CANONICAL_DATASETS.items():

    target = DATASET_TARGETS[
        dataset_id
    ]

    feature_columns = [
        column
        for column in df.columns
        if column != target
    ]

    numerical_columns = (
        df[feature_columns]
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_columns = [
        column
        for column in feature_columns
        if column not in numerical_columns
    ]


    # --------------------------------------------------------
    # Target type
    # --------------------------------------------------------

    target_is_numeric = (
        pd.api.types.is_numeric_dtype(
            df[target]
        )
    )


    DATASET_FEATURE_TYPES[
        dataset_id
    ] = {
        "target": target,
        "features": feature_columns,
        "numerical": numerical_columns,
        "categorical": categorical_columns,
        "target_numeric": target_is_numeric
    }


    for column in df.columns:

        if column == target:

            variable_role = "target"

        elif column in numerical_columns:

            variable_role = "numerical_feature"

        else:

            variable_role = "categorical_feature"


        DTYPE_RECORDS.append(
            {
                "dataset_id":
                    dataset_id,

                "column":
                    column,

                "pandas_dtype":
                    str(df[column].dtype),

                "variable_role":
                    variable_role,

                "unique_values":
                    int(
                        df[column]
                        .nunique(
                            dropna=True
                        )
                    )
            }
        )


DTYPE_DF = pd.DataFrame(
    DTYPE_RECORDS
)

display(DTYPE_DF)

print()
print("Data-type identification completed.")

,dataset_id,column,pandas_dtype,variable_role,unique_values
0,adult_income,age,int64,numerical_feature,73
1,adult_income,workclass,object,categorical_feature,8
2,adult_income,fnlwgt,int64,numerical_feature,21648
3,adult_income,education,object,categorical_feature,16
4,adult_income,education_num,int64,numerical_feature,16
...,...,...,...,...,...
75,diabetes_130us,metformin_rosiglitazone,object,categorical_feature,2
76,diabetes_130us,metformin_pioglitazone,object,categorical_feature,2
77,diabetes_130us,change,object,categorical_feature,2
78,diabetes_130us,diabetesMed,object,categorical_feature,2



Data-type identification completed.


In [9]:
# ============================================================
# CELL 02.3 — MISSING-VALUE REPRESENTATION
# ============================================================
#
# Purpose:
#   Standardize raw missing-value tokens.
#
# IMPORTANT:
#   No imputation is performed here.
#   Missingness generation is performed in Notebook 03.
# ============================================================

MISSING_TOKENS = {
    "",
    " ",
    "?",
    "NA",
    "N/A",
    "NULL",
    "null",
    "None",
    "none",
    "NaN",
    "nan"
}


MISSINGNESS_SUMMARY = []

for dataset_id, df in CANONICAL_DATASETS.items():

    dataset_missing_before = int(
        df.isna().sum().sum()
    )


    # --------------------------------------------------------
    # Standardize object/string columns
    # --------------------------------------------------------

    for column in df.columns:

        if (
            df[column].dtype == "object"
            or
            pd.api.types.is_string_dtype(
                df[column]
            )
        ):

            series = (
                df[column]
                .astype("string")
            )

            series = (
                series
                .str.strip()
            )

            series = series.mask(
                series.isin(
                    MISSING_TOKENS
                )
            )

            df[column] = series


    dataset_missing_after = int(
        df.isna().sum().sum()
    )


    MISSINGNESS_SUMMARY.append(
        {
            "dataset_id":
                dataset_id,

            "missing_before":
                dataset_missing_before,

            "missing_after":
                dataset_missing_after,

            "missing_values_standardized":
                dataset_missing_after
                - dataset_missing_before
        }
    )


MISSINGNESS_DF = pd.DataFrame(
    MISSINGNESS_SUMMARY
)

display(MISSINGNESS_DF)

print()
print(
    "Missing-value representation standardized."
)
print(
    "No imputation performed."
)

,dataset_id,missing_before,missing_after,missing_values_standardized
0,adult_income,4262,4262,0
1,bank_marketing,0,0,0
2,diabetes_130us,374017,374017,0



Missing-value representation standardized.
No imputation performed.


In [10]:
# ============================================================
# CELL 02.4 — CATEGORICAL NORMALIZATION
# ============================================================

CATEGORICAL_NORMALIZATION_LOG = []


for dataset_id, df in CANONICAL_DATASETS.items():

    type_info = DATASET_FEATURE_TYPES[
        dataset_id
    ]

    categorical_columns = (
        type_info["categorical"]
    )


    for column in categorical_columns:

        before = (
            df[column]
            .astype("string")
        )


        after = (
            before
            .str.strip()
        )


        # Empty strings become missing
        after = after.mask(
            after.eq("")
        )


        changed = int(
            (
                before.fillna(
                    "__NA__"
                )
                !=
                after.fillna(
                    "__NA__"
                )
            ).sum()
        )


        df[column] = after


        CATEGORICAL_NORMALIZATION_LOG.append(
            {
                "dataset_id":
                    dataset_id,

                "column":
                    column,

                "values_changed":
                    changed
            }
        )


CATEGORICAL_NORMALIZATION_DF = pd.DataFrame(
    CATEGORICAL_NORMALIZATION_LOG
)

display(
    CATEGORICAL_NORMALIZATION_DF
)

print()
print(
    "Categorical normalization completed."
)

,dataset_id,column,values_changed
0,adult_income,workclass,0
1,adult_income,education,0
2,adult_income,marital_status,0
3,adult_income,occupation,0
4,adult_income,relationship,0
5,adult_income,race,0
6,adult_income,sex,0
7,adult_income,native_country,0
8,bank_marketing,job,0
9,bank_marketing,marital,0



Categorical normalization completed.


In [11]:
# ============================================================
# CELL 02.5 — NUMERICAL NORMALIZATION
# ============================================================
#
# This cell converts numerical features to numeric dtype.
#
# No scaling is performed here because scaling parameters must
# be learned exclusively from the training partition.
# ============================================================

NUMERICAL_NORMALIZATION_LOG = []


for dataset_id, df in CANONICAL_DATASETS.items():

    numerical_columns = (
        DATASET_FEATURE_TYPES[
            dataset_id
        ]["numerical"]
    )


    for column in numerical_columns:

        original_dtype = str(
            df[column].dtype
        )


        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


        final_dtype = str(
            df[column].dtype
        )


        NUMERICAL_NORMALIZATION_LOG.append(
            {
                "dataset_id":
                    dataset_id,

                "column":
                    column,

                "original_dtype":
                    original_dtype,

                "final_dtype":
                    final_dtype,

                "conversion_performed":
                    original_dtype
                    != final_dtype
            }
        )


NUMERICAL_NORMALIZATION_DF = pd.DataFrame(
    NUMERICAL_NORMALIZATION_LOG
)

display(
    NUMERICAL_NORMALIZATION_DF
)

print()
print(
    "Numerical representation normalization completed."
)

,dataset_id,column,original_dtype,final_dtype,conversion_performed
0,adult_income,age,int64,int64,False
1,adult_income,fnlwgt,int64,int64,False
2,adult_income,education_num,int64,int64,False
3,adult_income,capital_gain,int64,int64,False
4,adult_income,capital_loss,int64,int64,False
5,adult_income,hours_per_week,int64,int64,False
6,bank_marketing,age,int64,int64,False
7,bank_marketing,balance,int64,int64,False
8,bank_marketing,day,int64,int64,False
9,bank_marketing,duration,int64,int64,False



Numerical representation normalization completed.


In [12]:
# ============================================================
# CELL 02.6 — FEATURE ENCODING CONFIGURATION
# ============================================================
#
# Encoding architecture is defined here.
#
# The actual encoder is fitted ONLY on the training split
# in Cell 02.10 to prevent leakage.
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


PREPROCESSING_CONFIG = {}

for dataset_id in DATASET_IDS:

    numerical_columns = (
        DATASET_FEATURE_TYPES[
            dataset_id
        ]["numerical"]
    )

    categorical_columns = (
        DATASET_FEATURE_TYPES[
            dataset_id
        ]["categorical"]
    )


    PREPROCESSING_CONFIG[
        dataset_id
    ] = {

        "numerical_features":
            numerical_columns,

        "categorical_features":
            categorical_columns,

        "encoding":
            "one_hot",

        "handle_unknown":
            "ignore",

        "drop":
            None
    }


print("=" * 100)
print("FEATURE ENCODING CONFIGURATION")
print("=" * 100)

for dataset_id, config in (
    PREPROCESSING_CONFIG.items()
):

    print()
    print(dataset_id)

    print(
        "Numerical features :",
        len(
            config[
                "numerical_features"
            ]
        )
    )

    print(
        "Categorical features:",
        len(
            config[
                "categorical_features"
            ]
        )
    )

    print(
        "Encoding           :",
        config["encoding"]
    )

print()
print(
    "Feature encoding architecture defined."
)
print(
    "Encoder fitting deferred until after train split."
)

FEATURE ENCODING CONFIGURATION

adult_income
Numerical features : 6
Categorical features: 8
Encoding           : one_hot

bank_marketing
Numerical features : 7
Categorical features: 9
Encoding           : one_hot

diabetes_130us
Numerical features : 11
Categorical features: 36
Encoding           : one_hot

Feature encoding architecture defined.
Encoder fitting deferred until after train split.


In [13]:
# ============================================================
# CELL 02.7 — TARGET IDENTIFICATION
# ============================================================

TARGET_VALIDATION_RECORDS = []


for dataset_id, df in CANONICAL_DATASETS.items():

    target = DATASET_TARGETS[
        dataset_id
    ]


    if target not in df.columns:

        raise RuntimeError(
            f"Target '{target}' missing from "
            f"{dataset_id}."
        )


    if df[target].isna().all():

        raise RuntimeError(
            f"Target '{target}' contains only "
            f"missing values in {dataset_id}."
        )


    target_type = (
        "numerical"
        if pd.api.types.is_numeric_dtype(
            df[target]
        )
        else "categorical"
    )


    TARGET_VALIDATION_RECORDS.append(
        {
            "dataset_id":
                dataset_id,

            "target":
                target,

            "target_type":
                target_type,

            "target_missing":
                int(
                    df[target].isna().sum()
                ),

            "target_unique":
                int(
                    df[target]
                    .nunique(
                        dropna=True
                    )
                ),

            "target_present":
                True
        }
    )


TARGET_DF = pd.DataFrame(
    TARGET_VALIDATION_RECORDS
)

display(TARGET_DF)

print()
print(
    "Target identification and validation completed."
)

,dataset_id,target,target_type,target_missing,target_unique,target_present
0,adult_income,income,categorical,0,2,True
1,bank_marketing,y,categorical,0,2,True
2,diabetes_130us,readmitted,categorical,0,3,True



Target identification and validation completed.


In [14]:
# ============================================================
# CELL 02.8 — TRAIN / VALIDATION / TEST SPLIT
# ============================================================

SEED = int(
    MASTER_CONFIG.get(
        "reproducibility",
        {}
    ).get(
        "seed",
        42
    )
)


TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15


if not np.isclose(
    TRAIN_SIZE
    + VALIDATION_SIZE
    + TEST_SIZE,
    1.0
):

    raise RuntimeError(
        "Train/validation/test proportions "
        "must sum to 1.0."
    )


RAW_SPLITS = {}

SPLIT_SUMMARY = []


for dataset_id, df in CANONICAL_DATASETS.items():

    target = DATASET_TARGETS[
        dataset_id
    ]


    # --------------------------------------------------------
    # Remove rows with missing target BEFORE splitting
    # --------------------------------------------------------
    #
    # This is NOT feature imputation.
    # A supervised experiment requires a valid target.
    #
    # Target missingness is not silently converted into a
    # feature value.
    # --------------------------------------------------------

    target_missing = df[target].isna()

    if target_missing.any():

        df = df.loc[
            ~target_missing
        ].copy()


    X = df.drop(
        columns=[target]
    )

    y = df[target]


    # --------------------------------------------------------
    # Initial 70 / 30 split
    # --------------------------------------------------------

    X_train, X_temp, y_train, y_temp = (
        train_test_split(
            X,
            y,
            test_size=(
                VALIDATION_SIZE
                + TEST_SIZE
            ),
            random_state=SEED,
            shuffle=True
        )
    )


    # --------------------------------------------------------
    # 15 / 15 split from remaining 30%
    # --------------------------------------------------------

    relative_test_size = (
        TEST_SIZE
        /
        (
            VALIDATION_SIZE
            + TEST_SIZE
        )
    )


    X_valid, X_test, y_valid, y_test = (
        train_test_split(
            X_temp,
            y_temp,
            test_size=relative_test_size,
            random_state=SEED,
            shuffle=True
        )
    )


    RAW_SPLITS[
        dataset_id
    ] = {

        "X_train":
            X_train.reset_index(
                drop=True
            ),

        "X_validation":
            X_valid.reset_index(
                drop=True
            ),

        "X_test":
            X_test.reset_index(
                drop=True
            ),

        "y_train":
            y_train.reset_index(
                drop=True
            ),

        "y_validation":
            y_valid.reset_index(
                drop=True
            ),

        "y_test":
            y_test.reset_index(
                drop=True
            )
    }


    SPLIT_SUMMARY.extend(
        [
            {
                "dataset_id":
                    dataset_id,

                "split":
                    "train",

                "rows":
                    len(X_train),

                "percentage":
                    len(X_train)
                    / len(df)
                    * 100
            },

            {
                "dataset_id":
                    dataset_id,

                "split":
                    "validation",

                "rows":
                    len(X_valid),

                "percentage":
                    len(X_valid)
                    / len(df)
                    * 100
            },

            {
                "dataset_id":
                    dataset_id,

                "split":
                    "test",

                "rows":
                    len(X_test),

                "percentage":
                    len(X_test)
                    / len(df)
                    * 100
            }
        ]
    )


SPLIT_SUMMARY_DF = pd.DataFrame(
    SPLIT_SUMMARY
)

display(
    SPLIT_SUMMARY_DF
)

print()
print(
    "Train/validation/test splitting completed."
)

,dataset_id,split,rows,percentage
0,adult_income,train,22792,69.997850
1,adult_income,validation,4884,14.999539
2,adult_income,test,4885,15.002610
3,bank_marketing,train,31647,69.998452
4,bank_marketing,validation,6782,15.000774
5,bank_marketing,test,6782,15.000774
6,diabetes_130us,train,71236,69.999803
7,diabetes_130us,validation,15265,15.000098
8,diabetes_130us,test,15265,15.000098



Train/validation/test splitting completed.


In [21]:
# ============================================================
# CELL 02.8 — TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 02.8 — TRAIN / VALIDATION / TEST SPLIT")
print("=" * 100)


# ------------------------------------------------------------
# Split configuration
# ------------------------------------------------------------

RANDOM_SEED = 42

TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

if not np.isclose(
    TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE,
    1.0
):
    raise ValueError(
        "Train/validation/test proportions must sum to 1.0."
    )


# ------------------------------------------------------------
# Containers
# ------------------------------------------------------------

RAW_SPLITS = {}


# ------------------------------------------------------------
# Process every canonical dataset
# ------------------------------------------------------------

for dataset_id, df in CANONICAL_DATASETS.items():

    target = DATASET_TARGETS[
        dataset_id
    ]

    print()
    print(f"Processing: {dataset_id}")


    # ========================================================
    # 1. CREATE IMMUTABLE OBSERVATION IDENTIFIERS
    # ========================================================

    # These IDs represent observations, not feature values.
    # They are NEVER used as model features.

    row_ids = np.arange(
        len(df),
        dtype=np.int64
    )


    # ========================================================
    # 2. FEATURE / TARGET SEPARATION
    # ========================================================

    X = df.drop(
        columns=[target]
    ).copy()

    y = df[target].copy()


    # ========================================================
    # 3. FIRST SPLIT
    #
    # 70% TRAIN
    # 30% TEMPORARY
    # ========================================================

    (
        X_train,
        X_temp,
        y_train,
        y_temp,
        row_id_train,
        row_id_temp
    ) = train_test_split(
        X,
        y,
        row_ids,
        test_size=(
            VALIDATION_SIZE
            +
            TEST_SIZE
        ),
        random_state=RANDOM_SEED,
        stratify=y
    )


    # ========================================================
    # 4. SECOND SPLIT
    #
    # TEMPORARY → VALIDATION + TEST
    # ========================================================

    validation_fraction_of_temp = (
        VALIDATION_SIZE
        /
        (
            VALIDATION_SIZE
            +
            TEST_SIZE
        )
    )


    (
        X_validation,
        X_test,
        y_validation,
        y_test,
        row_id_validation,
        row_id_test
    ) = train_test_split(
        X_temp,
        y_temp,
        row_id_temp,
        test_size=(
            1
            -
            validation_fraction_of_temp
        ),
        random_state=RANDOM_SEED,
        stratify=y_temp
    )


    # ========================================================
    # 5. RESET DATAFRAME INDICES
    # ========================================================

    X_train = X_train.reset_index(
        drop=True
    )

    X_validation = X_validation.reset_index(
        drop=True
    )

    X_test = X_test.reset_index(
        drop=True
    )

    y_train = y_train.reset_index(
        drop=True
    )

    y_validation = y_validation.reset_index(
        drop=True
    )

    y_test = y_test.reset_index(
        drop=True
    )


    # ========================================================
    # 6. NORMALIZE ROW-ID ARRAYS
    # ========================================================

    row_id_train = np.asarray(
        row_id_train,
        dtype=np.int64
    )

    row_id_validation = np.asarray(
        row_id_validation,
        dtype=np.int64
    )

    row_id_test = np.asarray(
        row_id_test,
        dtype=np.int64
    )


    # ========================================================
    # 7. BASIC PARTITION VALIDATION
    # ========================================================

    if (
        len(row_id_train)
        +
        len(row_id_validation)
        +
        len(row_id_test)
        !=
        len(df)
    ):
        raise RuntimeError(
            f"Partition size mismatch for "
            f"{dataset_id}."
        )


    train_ids = set(
        row_id_train.tolist()
    )

    validation_ids = set(
        row_id_validation.tolist()
    )

    test_ids = set(
        row_id_test.tolist()
    )


    if train_ids & validation_ids:
        raise RuntimeError(
            f"Train/validation overlap detected "
            f"for {dataset_id}."
        )

    if train_ids & test_ids:
        raise RuntimeError(
            f"Train/test overlap detected "
            f"for {dataset_id}."
        )

    if validation_ids & test_ids:
        raise RuntimeError(
            f"Validation/test overlap detected "
            f"for {dataset_id}."
        )


    # ========================================================
    # 8. STORE SPLITS
    # ========================================================

    RAW_SPLITS[
        dataset_id
    ] = {

        "X_train":
            X_train,

        "X_validation":
            X_validation,

        "X_test":
            X_test,

        "y_train":
            y_train,

        "y_validation":
            y_validation,

        "y_test":
            y_test,

        # -----------------------------------------------
        # Observation provenance identifiers
        # -----------------------------------------------

        "row_id_train":
            row_id_train,

        "row_id_validation":
            row_id_validation,

        "row_id_test":
            row_id_test
    }


    print(
        f"  Total rows      : {len(df):,}"
    )

    print(
        f"  Training rows   : {len(X_train):,}"
    )

    print(
        f"  Validation rows : {len(X_validation):,}"
    )

    print(
        f"  Test rows       : {len(X_test):,}"
    )

    print(
        "  Row-ID overlap  : 0"
    )


# ============================================================
# SPLIT SUMMARY
# ============================================================

SPLIT_SUMMARY = pd.DataFrame(
    [
        {
            "dataset_id":
                dataset_id,

            "total_rows":
                len(
                    CANONICAL_DATASETS[
                        dataset_id
                    ]
                ),

            "train_rows":
                len(
                    splits[
                        "X_train"
                    ]
                ),

            "validation_rows":
                len(
                    splits[
                        "X_validation"
                    ]
                ),

            "test_rows":
                len(
                    splits[
                        "X_test"
                    ]
                ),

            "train_fraction":
                len(
                    splits[
                        "X_train"
                    ]
                )
                /
                len(
                    CANONICAL_DATASETS[
                        dataset_id
                    ]
                ),

            "validation_fraction":
                len(
                    splits[
                        "X_validation"
                    ]
                )
                /
                len(
                    CANONICAL_DATASETS[
                        dataset_id
                    ]
                ),

            "test_fraction":
                len(
                    splits[
                        "X_test"
                    ]
                )
                /
                len(
                    CANONICAL_DATASETS[
                        dataset_id
                    ]
                ),

            "unique_train_ids":
                len(
                    set(
                        splits[
                            "row_id_train"
                        ].tolist()
                    )
                ),

            "unique_validation_ids":
                len(
                    set(
                        splits[
                            "row_id_validation"
                        ].tolist()
                    )
                ),

            "unique_test_ids":
                len(
                    set(
                        splits[
                            "row_id_test"
                        ].tolist()
                    )
                )
        }

        for dataset_id, splits
        in RAW_SPLITS.items()
    ]
)

display(
    SPLIT_SUMMARY
)


# ============================================================
# SAVE SPLIT SUMMARY
# ============================================================

SPLIT_SUMMARY_PATH = (
    NOTEBOOK02_RESULTS_DIR
    / "train_validation_test_split_summary.csv"
)

SPLIT_SUMMARY.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False
)


print()
print("=" * 100)
print("TRAIN / VALIDATION / TEST SPLITTING COMPLETED")
print("=" * 100)
print(
    f"Split summary saved:\n"
    f"{SPLIT_SUMMARY_PATH}"
)

CELL 02.8 — TRAIN / VALIDATION / TEST SPLIT

Processing: adult_income
  Total rows      : 32,561
  Training rows   : 22,792
  Validation rows : 4,884
  Test rows       : 4,885
  Row-ID overlap  : 0

Processing: bank_marketing
  Total rows      : 45,211
  Training rows   : 31,647
  Validation rows : 6,782
  Test rows       : 6,782
  Row-ID overlap  : 0

Processing: diabetes_130us
  Total rows      : 101,766
  Training rows   : 71,236
  Validation rows : 15,265
  Test rows       : 15,265
  Row-ID overlap  : 0


,dataset_id,total_rows,train_rows,validation_rows,test_rows,train_fraction,validation_fraction,test_fraction,unique_train_ids,unique_validation_ids,unique_test_ids
0,adult_income,32561,22792,4884,4885,0.699979,0.149995,0.150026,22792,4884,4885
1,bank_marketing,45211,31647,6782,6782,0.699985,0.150008,0.150008,31647,6782,6782
2,diabetes_130us,101766,71236,15265,15265,0.699998,0.150001,0.150001,71236,15265,15265



TRAIN / VALIDATION / TEST SPLITTING COMPLETED
Split summary saved:
/content/air_llm_drive/MyDrive/AIR_LLM_Research/results/processed/notebook_02/train_validation_test_split_summary.csv


In [22]:
# ============================================================
# CELL 02.9 — STRATIFICATION VALIDATION
# ============================================================

STRATIFICATION_RECORDS = []


for dataset_id, splits in RAW_SPLITS.items():

    target = DATASET_TARGETS[
        dataset_id
    ]


    full_distribution = (
        CANONICAL_DATASETS[
            dataset_id
        ]
        .loc[
            lambda x:
            x[target].notna(),
            target
        ]
        .value_counts(
            normalize=True
        )
    )


    for split_name in [
        "train",
        "validation",
        "test"
    ]:

        y = splits[
            f"y_{split_name}"
        ]


        split_distribution = (
            y.value_counts(
                normalize=True
            )
        )


        common_classes = (
            full_distribution.index
            .intersection(
                split_distribution.index
            )
        )


        max_distribution_difference = (
            (
                full_distribution[
                    common_classes
                ]
                -
                split_distribution[
                    common_classes
                ]
            )
            .abs()
            .max()
        )


        STRATIFICATION_RECORDS.append(
            {
                "dataset_id":
                    dataset_id,

                "split":
                    split_name,

                "max_class_distribution_difference":
                    float(
                        max_distribution_difference
                    ),

                "stratification_valid":
                    bool(
                        max_distribution_difference
                        <= 0.05
                    )
            }
        )


STRATIFICATION_DF = pd.DataFrame(
    STRATIFICATION_RECORDS
)

display(
    STRATIFICATION_DF
)


if not STRATIFICATION_DF[
    "stratification_valid"
].all():

    raise RuntimeError(
        "Stratification validation failed."
    )


print()
print(
    "Stratification validation: PASSED"
)

,dataset_id,split,max_class_distribution_difference,stratification_valid
0,adult_income,train,0.000021,True
1,adult_income,validation,0.000023,True
2,adult_income,test,0.000073,True
3,bank_marketing,train,0.000007,True
4,bank_marketing,validation,0.000090,True
5,bank_marketing,test,0.000058,True
6,diabetes_130us,train,0.000006,True
7,diabetes_130us,validation,0.000037,True
8,diabetes_130us,test,0.000043,True



Stratification validation: PASSED


In [23]:
# ============================================================
# CELL 02.10 — PREPROCESSING PIPELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

import numpy as np
import pandas as pd


print("=" * 100)
print("02.10 — PREPROCESSING PIPELINE")
print("=" * 100)


# ============================================================
# 1. VALIDATE REQUIRED INPUTS
# ============================================================

required_objects = [
    "RAW_SPLITS",
    "DATASET_FEATURE_TYPES"
]

for object_name in required_objects:

    if object_name not in globals():

        raise RuntimeError(
            f"{object_name} is not defined.\n"
            f"Run the preceding Notebook 02 cells before 02.10."
        )


# ============================================================
# 2. MISSING-VALUE NORMALIZATION
# ============================================================
#
# sklearn's SimpleImputer works reliably when missing values
# are represented consistently as np.nan.
#
# AIR-LLM therefore converts:
#
#   pd.NA
#   None
#   NaN
#   common textual missing markers
#
# into np.nan BEFORE the sklearn preprocessing pipeline.
#
# This does NOT impute the data.
#
# Actual imputation remains inside the sklearn pipeline and
# is fitted ONLY on X_train.
# ============================================================

MISSING_VALUE_MARKERS = {
    "",
    " ",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "None",
    "none",
    "?",
    "unknown",
    "Unknown"
}


def normalize_missing_values(df):
    """
    Convert heterogeneous missing-value representations into
    np.nan without performing statistical imputation.
    """

    result = df.copy()

    # --------------------------------------------------------
    # Convert pandas nullable values safely
    # --------------------------------------------------------

    for column in result.columns:

        series = result[column]

        # ----------------------------------------------------
        # Object/string/categorical columns
        # ----------------------------------------------------

        if (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or pd.api.types.is_categorical_dtype(series)
        ):

            # Convert to object first so np.nan can be inserted
            # safely into categorical/string-backed columns.
            series = series.astype(object)

            series = series.replace(
                list(MISSING_VALUE_MARKERS),
                np.nan
            )

            # Explicitly normalize pd.NA / None
            series = series.where(
                pd.notna(series),
                np.nan
            )

            result[column] = series

        # ----------------------------------------------------
        # Numerical columns
        # ----------------------------------------------------

        else:

            result[column] = pd.to_numeric(
                series,
                errors="coerce"
            )

    return result


# ============================================================
# 3. CONTAINER INITIALIZATION
# ============================================================

PREPROCESSING_PIPELINES = {}

PROCESSED_SPLITS = {}

PREPROCESSING_METADATA = {}

PREPROCESSING_SUMMARY_ROWS = []


# ============================================================
# 4. PROCESS EACH DATASET
# ============================================================

for dataset_id, splits in RAW_SPLITS.items():

    print()
    print("-" * 100)
    print(f"Processing dataset: {dataset_id}")
    print("-" * 100)


    # ========================================================
    # 4.1 COLUMN TYPES
    # ========================================================

    numerical_columns = list(
        DATASET_FEATURE_TYPES[
            dataset_id
        ]["numerical"]
    )

    categorical_columns = list(
        DATASET_FEATURE_TYPES[
            dataset_id
        ]["categorical"]
    )


    # ========================================================
    # 4.2 COPY RAW EXPERIMENTAL SPLITS
    # ========================================================

    X_train = splits[
        "X_train"
    ].copy()

    X_validation = splits[
        "X_validation"
    ].copy()

    X_test = splits[
        "X_test"
    ].copy()


    # ========================================================
    # 4.3 NORMALIZE MISSING REPRESENTATIONS
    # ========================================================

    X_train = normalize_missing_values(
        X_train
    )

    X_validation = normalize_missing_values(
        X_validation
    )

    X_test = normalize_missing_values(
        X_test
    )


    # ========================================================
    # 4.4 VERIFY COLUMN DEFINITIONS
    # ========================================================

    expected_feature_columns = (
        numerical_columns
        + categorical_columns
    )

    missing_columns = [
        column
        for column in expected_feature_columns
        if column not in X_train.columns
    ]

    if missing_columns:

        raise RuntimeError(
            f"{dataset_id}: feature columns missing "
            f"from training data:\n"
            f"{missing_columns}"
        )


    # ========================================================
    # 4.5 REMOVE TARGET FROM FEATURES IF NECESSARY
    # ========================================================

    target = DATASET_TARGETS[
        dataset_id
    ]

    for dataframe_name, dataframe in [
        ("X_train", X_train),
        ("X_validation", X_validation),
        ("X_test", X_test)
    ]:

        if target in dataframe.columns:

            raise RuntimeError(
                f"DATA LEAKAGE DETECTED in {dataset_id}: "
                f"target '{target}' appears in {dataframe_name}."
            )


    # ========================================================
    # 4.6 NUMERICAL PIPELINE
    # ========================================================

    numerical_pipeline = Pipeline(
        steps=[

            (
                "imputer",

                SimpleImputer(
                    strategy="median",
                    missing_values=np.nan
                )
            )

        ]
    )


    # ========================================================
    # 4.7 CATEGORICAL PIPELINE
    # ========================================================

    categorical_pipeline = Pipeline(
        steps=[

            (
                "imputer",

                SimpleImputer(
                    strategy="most_frequent",
                    missing_values=np.nan
                )
            ),

            (
                "encoder",

                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                    dtype=np.float64
                )
            )

        ]
    )


    # ========================================================
    # 4.8 COMBINED PREPROCESSING PIPELINE
    # ========================================================

    transformers = []


    if numerical_columns:

        transformers.append(
            (
                "numerical",
                numerical_pipeline,
                numerical_columns
            )
        )


    if categorical_columns:

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        )


    if not transformers:

        raise RuntimeError(
            f"No usable feature columns found for "
            f"{dataset_id}."
        )


    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


    # ========================================================
    # 4.9 FIT ONLY ON TRAINING DATA
    # ========================================================

    preprocessor.fit(
        X_train
    )


    # ========================================================
    # 4.10 TRANSFORM TRAINING DATA
    # ========================================================

    X_train_processed = (
        preprocessor.transform(
            X_train
        )
    )


    # ========================================================
    # 4.11 TRANSFORM VALIDATION DATA
    # ========================================================

    X_validation_processed = (
        preprocessor.transform(
            X_validation
        )
    )


    # ========================================================
    # 4.12 TRANSFORM TEST DATA
    # ========================================================

    X_test_processed = (
        preprocessor.transform(
            X_test
        )
    )


    # ========================================================
    # 4.13 FORCE NUMERICAL FLOAT64 OUTPUT
    # ========================================================

    X_train_processed = np.asarray(
        X_train_processed,
        dtype=np.float64
    )

    X_validation_processed = np.asarray(
        X_validation_processed,
        dtype=np.float64
    )

    X_test_processed = np.asarray(
        X_test_processed,
        dtype=np.float64
    )


    # ========================================================
    # 4.14 CHECK FOR INVALID VALUES
    # ========================================================

    for split_name, matrix in [

        (
            "X_train",
            X_train_processed
        ),

        (
            "X_validation",
            X_validation_processed
        ),

        (
            "X_test",
            X_test_processed
        )

    ]:

        if not np.isfinite(
            matrix
        ).all():

            raise RuntimeError(
                f"{dataset_id}: non-finite values detected "
                f"in processed {split_name}."
            )


    # ========================================================
    # 4.15 FEATURE NAMES
    # ========================================================

    feature_names = (
        preprocessor
        .get_feature_names_out()
        .tolist()
    )


    # ========================================================
    # 4.16 FEATURE-NAME VALIDATION
    # ========================================================

    if len(feature_names) != (
        X_train_processed.shape[1]
    ):

        raise RuntimeError(
            f"Feature-name mismatch for {dataset_id}.\n"
            f"Feature names: {len(feature_names)}\n"
            f"Processed features: "
            f"{X_train_processed.shape[1]}"
        )


    if len(
        set(feature_names)
    ) != len(feature_names):

        raise RuntimeError(
            f"Duplicate processed feature names detected "
            f"for {dataset_id}."
        )


    # ========================================================
    # 4.17 TARGET ARRAYS
    # ========================================================

    y_train = np.asarray(
        splits[
            "y_train"
        ]
    )

    y_validation = np.asarray(
        splits[
            "y_validation"
        ]
    )

    y_test = np.asarray(
        splits[
            "y_test"
        ]
    )


    # ========================================================
    # 4.18 ROW-COUNT VALIDATION
    # ========================================================

    if len(y_train) != (
        X_train_processed.shape[0]
    ):

        raise RuntimeError(
            f"{dataset_id}: training X/y row mismatch."
        )


    if len(y_validation) != (
        X_validation_processed.shape[0]
    ):

        raise RuntimeError(
            f"{dataset_id}: validation X/y row mismatch."
        )


    if len(y_test) != (
        X_test_processed.shape[0]
    ):

        raise RuntimeError(
            f"{dataset_id}: test X/y row mismatch."
        )


    # ========================================================
    # 4.19 STORE FITTED PIPELINE
    # ========================================================

    PREPROCESSING_PIPELINES[
        dataset_id
    ] = preprocessor


    # ========================================================
    # 4.20 STORE PROCESSED DATA
    # ========================================================

    PROCESSED_SPLITS[
        dataset_id
    ] = {

        "X_train":
            X_train_processed,

        "X_validation":
            X_validation_processed,

        "X_test":
            X_test_processed,

        "y_train":
            y_train,

        "y_validation":
            y_validation,

        "y_test":
            y_test,

        "feature_names":
            feature_names
    }


    # ========================================================
    # 4.21 STORE PREPROCESSING METADATA
    # ========================================================

    PREPROCESSING_METADATA[
        dataset_id
    ] = {

        "numerical_columns":
            numerical_columns,

        "categorical_columns":
            categorical_columns,

        "processed_feature_count":
            len(feature_names),

        "missing_value_strategy_numerical":
            "median",

        "missing_value_strategy_categorical":
            "most_frequent",

        "encoding":
            "OneHotEncoder",

        "unknown_category_handling":
            "ignore",

        "fit_partition":
            "training_only",

        "output_dtype":
            "float64"
    }


    # ========================================================
    # 4.22 SUMMARY
    # ========================================================

    PREPROCESSING_SUMMARY_ROWS.append(
        {

            "dataset_id":
                dataset_id,

            "train_rows":
                X_train_processed.shape[0],

            "validation_rows":
                X_validation_processed.shape[0],

            "test_rows":
                X_test_processed.shape[0],

            "original_features":
                len(expected_feature_columns),

            "numerical_features":
                len(numerical_columns),

            "categorical_features":
                len(categorical_columns),

            "processed_features":
                X_train_processed.shape[1],

            "output_dtype":
                str(
                    X_train_processed.dtype
                ),

            "train_finite":
                bool(
                    np.isfinite(
                        X_train_processed
                    ).all()
                ),

            "validation_finite":
                bool(
                    np.isfinite(
                        X_validation_processed
                    ).all()
                ),

            "test_finite":
                bool(
                    np.isfinite(
                        X_test_processed
                    ).all()
                )
        }
    )


    print(
        f"✓ {dataset_id}: "
        f"{X_train_processed.shape[0]:,} train rows | "
        f"{X_validation_processed.shape[0]:,} validation rows | "
        f"{X_test_processed.shape[0]:,} test rows | "
        f"{X_train_processed.shape[1]:,} processed features"
    )


# ============================================================
# 5. FINAL SUMMARY
# ============================================================

PREPROCESSING_SUMMARY = pd.DataFrame(
    PREPROCESSING_SUMMARY_ROWS
)

display(
    PREPROCESSING_SUMMARY
)


# ============================================================
# 6. SAVE PREPROCESSING SUMMARY
# ============================================================

if "NOTEBOOK02_RESULTS_DIR" not in globals():

    NOTEBOOK02_RESULTS_DIR = (
        PROJECT_ROOT
        / "results"
        / "processed"
        / "notebook_02"
    )

NOTEBOOK02_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PREPROCESSING_SUMMARY_PATH = (
    NOTEBOOK02_RESULTS_DIR
    / "preprocessing_summary.csv"
)

PREPROCESSING_SUMMARY.to_csv(
    PREPROCESSING_SUMMARY_PATH,
    index=False
)


# ============================================================
# 7. FINAL VALIDATION
# ============================================================

if set(
    PROCESSED_SPLITS.keys()
) != set(
    RAW_SPLITS.keys()
):

    raise RuntimeError(
        "Processed split coverage does not match "
        "RAW_SPLITS."
    )


if set(
    PREPROCESSING_PIPELINES.keys()
) != set(
    RAW_SPLITS.keys()
):

    raise RuntimeError(
        "Preprocessing pipeline coverage does not match "
        "RAW_SPLITS."
    )


print()
print("=" * 100)
print("02.10 PREPROCESSING PIPELINE COMPLETE")
print("=" * 100)

print(
    "✓ Missing representations normalized to np.nan"
)

print(
    "✓ Numerical missing values handled by median imputation"
)

print(
    "✓ Categorical missing values handled by most-frequent imputation"
)

print(
    "✓ Categorical variables one-hot encoded"
)

print(
    "✓ Pipelines fitted on TRAINING DATA ONLY"
)

print(
    "✓ Validation and test sets transformed using training-fitted pipelines"
)

print(
    "✓ All processed matrices validated as finite float64"
)

print(
    "✓ Feature names validated"
)

print(
    "✓ X/y row alignment validated"
)

print()
print(
    f"Summary saved:\n{PREPROCESSING_SUMMARY_PATH}"
)

print()
print(
    "READY FOR CELL 02.11 — LEAKAGE VALIDATION"
)

print("=" * 100)

02.10 — PREPROCESSING PIPELINE

----------------------------------------------------------------------------------------------------
Processing dataset: adult_income
----------------------------------------------------------------------------------------------------
✓ adult_income: 22,792 train rows | 4,884 validation rows | 4,885 test rows | 105 processed features

----------------------------------------------------------------------------------------------------
Processing dataset: bank_marketing
----------------------------------------------------------------------------------------------------
✓ bank_marketing: 31,647 train rows | 6,782 validation rows | 6,782 test rows | 47 processed features

----------------------------------------------------------------------------------------------------
Processing dataset: diabetes_130us
----------------------------------------------------------------------------------------------------
✓ diabetes_130us: 71,236 train rows | 15,265 validatio

,dataset_id,train_rows,validation_rows,test_rows,original_features,numerical_features,categorical_features,processed_features,output_dtype,train_finite,validation_finite,test_finite
0,adult_income,22792,4884,4885,14,6,8,105,float64,True,True,True
1,bank_marketing,31647,6782,6782,16,7,9,47,float64,True,True,True
2,diabetes_130us,71236,15265,15265,47,11,36,2322,float64,True,True,True



02.10 PREPROCESSING PIPELINE COMPLETE
✓ Missing representations normalized to np.nan
✓ Numerical missing values handled by median imputation
✓ Categorical missing values handled by most-frequent imputation
✓ Categorical variables one-hot encoded
✓ Pipelines fitted on TRAINING DATA ONLY
✓ Validation and test sets transformed using training-fitted pipelines
✓ All processed matrices validated as finite float64
✓ Feature names validated
✓ X/y row alignment validated

Summary saved:
/content/air_llm_drive/MyDrive/AIR_LLM_Research/results/processed/notebook_02/preprocessing_summary.csv

READY FOR CELL 02.11 — LEAKAGE VALIDATION


In [24]:
# ============================================================
# CELL 02.11 — LEAKAGE VALIDATION
# ============================================================

import hashlib
import json
import numpy as np
import pandas as pd

print("=" * 100)
print("CELL 02.11 — LEAKAGE VALIDATION")
print("=" * 100)

LEAKAGE_RECORDS = []


# ------------------------------------------------------------
# Stable row identity helper
# ------------------------------------------------------------

def create_stable_row_ids(df):
    """
    Create deterministic row identifiers from the complete
    canonical row content.

    These identifiers are used only for experimental integrity
    validation and are never supplied as model features.
    """

    normalized = df.copy()

    normalized = normalized.astype(
        "string"
    ).fillna(
        "<NA>"
    )

    row_ids = (
        pd.util.hash_pandas_object(
            normalized,
            index=False
        )
        .astype("uint64")
        .astype(str)
    )

    return row_ids


# ------------------------------------------------------------
# Validate every dataset
# ------------------------------------------------------------

for dataset_id, splits in RAW_SPLITS.items():

    target = DATASET_TARGETS[
        dataset_id
    ]

    preprocessor = PREPROCESSING_PIPELINES[
        dataset_id
    ]


    # ========================================================
    # 1. TARGET EXCLUSION
    # ========================================================

    target_in_train_features = (
        target in splits["X_train"].columns
    )

    target_in_validation_features = (
        target in splits["X_validation"].columns
    )

    target_in_test_features = (
        target in splits["X_test"].columns
    )


    # ========================================================
    # 2. ROW ID VALIDATION
    # ========================================================

    if "row_id_train" in splits:

        train_row_ids = set(
            splits["row_id_train"]
        )

        validation_row_ids = set(
            splits["row_id_validation"]
        )

        test_row_ids = set(
            splits["row_id_test"]
        )

    else:

        # ----------------------------------------------------
        # Backward-compatible reconstruction.
        #
        # This should only be necessary if row IDs were not
        # explicitly retained by the splitting cell.
        # ----------------------------------------------------

        train_source = splits["X_train"].copy()
        validation_source = splits["X_validation"].copy()
        test_source = splits["X_test"].copy()

        train_row_ids = set(
            create_stable_row_ids(
                train_source
            )
        )

        validation_row_ids = set(
            create_stable_row_ids(
                validation_source
            )
        )

        test_row_ids = set(
            create_stable_row_ids(
                test_source
            )
        )


    # ========================================================
    # 3. TRUE PARTITION OVERLAP
    # ========================================================

    train_validation_overlap = len(
        train_row_ids
        &
        validation_row_ids
    )

    train_test_overlap = len(
        train_row_ids
        &
        test_row_ids
    )

    validation_test_overlap = len(
        validation_row_ids
        &
        test_row_ids
    )


    # ========================================================
    # 4. PARTITION SIZE VALIDATION
    # ========================================================

    train_size = len(
        train_row_ids
    )

    validation_size = len(
        validation_row_ids
    )

    test_size = len(
        test_row_ids
    )


    partition_sizes_valid = all(
        size > 0
        for size in [
            train_size,
            validation_size,
            test_size
        ]
    )


    # ========================================================
    # 5. TOTAL PARTITION COVERAGE
    # ========================================================

    total_partition_rows = (
        train_size
        +
        validation_size
        +
        test_size
    )

    partition_union_size = len(
        train_row_ids
        |
        validation_row_ids
        |
        test_row_ids
    )

    partitions_disjoint = (
        train_validation_overlap == 0
        and
        train_test_overlap == 0
        and
        validation_test_overlap == 0
    )


    # ========================================================
    # 6. PREPROCESSOR FIT VALIDATION
    # ========================================================

    preprocessor_fitted = hasattr(
        preprocessor,
        "transformers_"
    )


    # ========================================================
    # 7. FEATURE-DIMENSION CONSISTENCY
    # ========================================================

    feature_count_train = (
        PROCESSED_SPLITS[
            dataset_id
        ]["X_train"].shape[1]
    )

    feature_count_validation = (
        PROCESSED_SPLITS[
            dataset_id
        ]["X_validation"].shape[1]
    )

    feature_count_test = (
        PROCESSED_SPLITS[
            dataset_id
        ]["X_test"].shape[1]
    )

    feature_dimensions_consistent = (
        feature_count_train
        ==
        feature_count_validation
        ==
        feature_count_test
    )


    # ========================================================
    # 8. PROCESSED ROW COUNT CONSISTENCY
    # ========================================================

    processed_row_counts_consistent = (
        PROCESSED_SPLITS[
            dataset_id
        ]["X_train"].shape[0]
        ==
        train_size
        and
        PROCESSED_SPLITS[
            dataset_id
        ]["X_validation"].shape[0]
        ==
        validation_size
        and
        PROCESSED_SPLITS[
            dataset_id
        ]["X_test"].shape[0]
        ==
        test_size
    )


    # ========================================================
    # 9. FINAL LEAKAGE DECISION
    # ========================================================

    leakage_free = all(
        [
            not target_in_train_features,
            not target_in_validation_features,
            not target_in_test_features,

            partition_sizes_valid,

            partitions_disjoint,

            preprocessor_fitted,

            feature_dimensions_consistent,

            processed_row_counts_consistent
        ]
    )


    # ========================================================
    # 10. RECORD
    # ========================================================

    LEAKAGE_RECORDS.append(
        {
            "dataset_id":
                dataset_id,

            "train_rows":
                train_size,

            "validation_rows":
                validation_size,

            "test_rows":
                test_size,

            "total_partition_rows":
                total_partition_rows,

            "unique_partition_rows":
                partition_union_size,

            "target_in_train_features":
                target_in_train_features,

            "target_in_validation_features":
                target_in_validation_features,

            "target_in_test_features":
                target_in_test_features,

            "train_validation_row_overlap":
                train_validation_overlap,

            "train_test_row_overlap":
                train_test_overlap,

            "validation_test_row_overlap":
                validation_test_overlap,

            "partitions_disjoint":
                partitions_disjoint,

            "partition_sizes_valid":
                partition_sizes_valid,

            "preprocessor_fitted":
                preprocessor_fitted,

            "feature_dimensions_consistent":
                feature_dimensions_consistent,

            "processed_row_counts_consistent":
                processed_row_counts_consistent,

            "leakage_free":
                leakage_free
        }
    )


# ------------------------------------------------------------
# Create validation dataframe
# ------------------------------------------------------------

LEAKAGE_DF = pd.DataFrame(
    LEAKAGE_RECORDS
)

display(
    LEAKAGE_DF
)


# ------------------------------------------------------------
# Save leakage validation report
# ------------------------------------------------------------

LEAKAGE_REPORT_PATH = (
    NOTEBOOK02_RESULTS_DIR
    / "leakage_validation.csv"
)

LEAKAGE_DF.to_csv(
    LEAKAGE_REPORT_PATH,
    index=False
)


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

if not LEAKAGE_DF[
    "leakage_free"
].all():

    failed = (
        LEAKAGE_DF.loc[
            ~LEAKAGE_DF[
                "leakage_free"
            ],
            "dataset_id"
        ]
        .tolist()
    )

    raise RuntimeError(
        "DATA LEAKAGE VALIDATION FAILED.\n"
        f"Datasets: {failed}\n"
        f"Report: {LEAKAGE_REPORT_PATH}"
    )


print()
print("=" * 100)
print("LEAKAGE VALIDATION: PASSED")
print("=" * 100)
print()
print(
    f"Validation report saved:\n"
    f"{LEAKAGE_REPORT_PATH}"
)

CELL 02.11 — LEAKAGE VALIDATION


,dataset_id,train_rows,validation_rows,test_rows,total_partition_rows,unique_partition_rows,target_in_train_features,target_in_validation_features,target_in_test_features,train_validation_row_overlap,train_test_row_overlap,validation_test_row_overlap,partitions_disjoint,partition_sizes_valid,preprocessor_fitted,feature_dimensions_consistent,processed_row_counts_consistent,leakage_free
0,adult_income,22792,4884,4885,32561,32561,False,False,False,0,0,0,True,True,True,True,True,True
1,bank_marketing,31647,6782,6782,45211,45211,False,False,False,0,0,0,True,True,True,True,True,True
2,diabetes_130us,71236,15265,15265,101766,101766,False,False,False,0,0,0,True,True,True,True,True,True



LEAKAGE VALIDATION: PASSED

Validation report saved:
/content/air_llm_drive/MyDrive/AIR_LLM_Research/results/processed/notebook_02/leakage_validation.csv


In [25]:

# ============================================================
# CELL 02.12 — SAVE EXPERIMENTAL DATASETS
# ============================================================

import joblib


# ------------------------------------------------------------
# Create dedicated directories
# ------------------------------------------------------------

RAW_SPLIT_DIR = (
    PREPROCESSING_DIR
    / "raw_splits"
)

PROCESSED_SPLIT_DIR = (
    PREPROCESSING_DIR
    / "encoded_splits"
)

PIPELINE_DIR = (
    PREPROCESSING_DIR
    / "pipelines"
)

METADATA_DIR = (
    PREPROCESSING_DIR
    / "metadata"
)


for directory in [
    RAW_SPLIT_DIR,
    PROCESSED_SPLIT_DIR,
    PIPELINE_DIR,
    METADATA_DIR
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# Save datasets
# ------------------------------------------------------------

SAVE_MANIFEST = []


for dataset_id in DATASET_IDS:

    raw_splits = RAW_SPLITS[
        dataset_id
    ]

    processed_splits = PROCESSED_SPLITS[
        dataset_id
    ]


    # ========================================================
    # RAW SPLITS
    # ========================================================

    for split_name in [
        "train",
        "validation",
        "test"
    ]:

        X = raw_splits[
            f"X_{split_name}"
        ]

        y = raw_splits[
            f"y_{split_name}"
        ]


        raw_output = X.copy()

        raw_output[
            DATASET_TARGETS[
                dataset_id
            ]
        ] = y


        raw_path = (
            RAW_SPLIT_DIR
            / f"{dataset_id}_{split_name}.csv"
        )


        raw_output.to_csv(
            raw_path,
            index=False,
            encoding="utf-8"
        )


    # ========================================================
    # PROCESSED / ENCODED SPLITS
    # ========================================================

    feature_names = (
        processed_splits[
            "feature_names"
        ]
    )


    for split_name in [
        "train",
        "validation",
        "test"
    ]:

        X = processed_splits[
            f"X_{split_name}"
        ]

        y = processed_splits[
            f"y_{split_name}"
        ]


        processed_df = pd.DataFrame(
            X,
            columns=feature_names
        )


        processed_df[
            DATASET_TARGETS[
                dataset_id
            ]
        ] = y


        processed_path = (
            PROCESSED_SPLIT_DIR
            / f"{dataset_id}_{split_name}_encoded.csv"
        )


        processed_df.to_csv(
            processed_path,
            index=False,
            encoding="utf-8"
        )


    # ========================================================
    # SAVE FITTED PREPROCESSOR
    # ========================================================

    pipeline_path = (
        PIPELINE_DIR
        / f"{dataset_id}_preprocessor.joblib"
    )


    joblib.dump(
        PREPROCESSING_PIPELINES[
            dataset_id
        ],
        pipeline_path
    )


    # ========================================================
    # SAVE FEATURE NAMES
    # ========================================================

    feature_names_path = (
        METADATA_DIR
        / f"{dataset_id}_feature_names.json"
    )


    with open(
        feature_names_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            feature_names,
            file,
            indent=2,
            ensure_ascii=False
        )


    # ========================================================
    # SAVE DATASET METADATA
    # ========================================================

    metadata = {

        "dataset_id":
            dataset_id,

        "target":
            DATASET_TARGETS[
                dataset_id
            ],

        "seed":
            SEED,

        "train_fraction":
            TRAIN_SIZE,

        "validation_fraction":
            VALIDATION_SIZE,

        "test_fraction":
            TEST_SIZE,

        "numerical_features":
            DATASET_FEATURE_TYPES[
                dataset_id
            ]["numerical"],

        "categorical_features":
            DATASET_FEATURE_TYPES[
                dataset_id
            ]["categorical"],

        "encoding":
            "one_hot",

        "preprocessor_fit_on":
            "training_split_only",

        "processed_feature_count":
            len(feature_names)
    }


    metadata_path = (
        METADATA_DIR
        / f"{dataset_id}_preprocessing_metadata.json"
    )


    with open(
        metadata_path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            metadata,
            file,
            indent=2,
            ensure_ascii=False
        )


    SAVE_MANIFEST.append(
        {
            "dataset_id":
                dataset_id,

            "raw_train":
                str(
                    RAW_SPLIT_DIR
                    / f"{dataset_id}_train.csv"
                ),

            "raw_validation":
                str(
                    RAW_SPLIT_DIR
                    / f"{dataset_id}_validation.csv"
                ),

            "raw_test":
                str(
                    RAW_SPLIT_DIR
                    / f"{dataset_id}_test.csv"
                ),

            "encoded_train":
                str(
                    PROCESSED_SPLIT_DIR
                    / f"{dataset_id}_train_encoded.csv"
                ),

            "encoded_validation":
                str(
                    PROCESSED_SPLIT_DIR
                    / f"{dataset_id}_validation_encoded.csv"
                ),

            "encoded_test":
                str(
                    PROCESSED_SPLIT_DIR
                    / f"{dataset_id}_test_encoded.csv"
                ),

            "preprocessor":
                str(
                    pipeline_path
                )
        }
    )


SAVE_MANIFEST_DF = pd.DataFrame(
    SAVE_MANIFEST
)

display(
    SAVE_MANIFEST_DF
)


# ------------------------------------------------------------
# Save Notebook 02 manifest
# ------------------------------------------------------------

NOTEBOOK02_MANIFEST_PATH = (
    ARTIFACTS_DIR
    / "notebook_02_manifest.json"
)


with open(
    NOTEBOOK02_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "notebook":
                "02_Data_Preprocessing_and_Splitting",

            "project":
                "AIR-LLM",

            "project_version":
                "1.0",

            "experiment_version":
                "AIR-LLM-v1",

            "configuration":
                "CONFIG-v1",

            "seed":
                SEED,

            "datasets":
                DATASET_IDS,

            "targets":
                DATASET_TARGETS,

            "train_fraction":
                TRAIN_SIZE,

            "validation_fraction":
                VALIDATION_SIZE,

            "test_fraction":
                TEST_SIZE,

            "preprocessing_fit":
                "training_only",

            "encoding":
                "one_hot",

            "manifest":
                SAVE_MANIFEST
        },
        file,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 100)
print("NOTEBOOK 02 DATA SAVING COMPLETE")
print("=" * 100)

print(
    f"Raw splits      : {RAW_SPLIT_DIR}"
)

print(
    f"Encoded splits  : {PROCESSED_SPLIT_DIR}"
)

print(
    f"Pipelines       : {PIPELINE_DIR}"
)

print(
    f"Metadata        : {METADATA_DIR}"
)

print(
    f"Manifest        : {NOTEBOOK02_MANIFEST_PATH}"
)

,dataset_id,raw_train,raw_validation,raw_test,encoded_train,encoded_validation,encoded_test,preprocessor
0,adult_income,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...
1,bank_marketing,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...
2,diabetes_130us,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...,/content/air_llm_drive/MyDrive/AIR_LLM_Researc...



NOTEBOOK 02 DATA SAVING COMPLETE
Raw splits      : /content/air_llm_drive/MyDrive/AIR_LLM_Research/experiments/preprocessing/raw_splits
Encoded splits  : /content/air_llm_drive/MyDrive/AIR_LLM_Research/experiments/preprocessing/encoded_splits
Pipelines       : /content/air_llm_drive/MyDrive/AIR_LLM_Research/experiments/preprocessing/pipelines
Metadata        : /content/air_llm_drive/MyDrive/AIR_LLM_Research/experiments/preprocessing/metadata
Manifest        : /content/air_llm_drive/MyDrive/AIR_LLM_Research/artifacts/notebook_02_manifest.json


In [26]:
# ============================================================
# CELL 02.12 — FINAL NOTEBOOK 02 VALIDATION
# ============================================================

FINAL_VALIDATION_RECORDS = []


for dataset_id in DATASET_IDS:

    raw = RAW_SPLITS[
        dataset_id
    ]

    processed = PROCESSED_SPLITS[
        dataset_id
    ]

    target = DATASET_TARGETS[
        dataset_id
    ]


    checks = {

        "train_exists":
            len(raw["X_train"]) > 0,

        "validation_exists":
            len(raw["X_validation"]) > 0,

        "test_exists":
            len(raw["X_test"]) > 0,

        "target_exists":
            all(
                key in raw
                for key in [
                    "y_train",
                    "y_validation",
                    "y_test"
                ]
            ),

        "feature_dimensions_match":
            (
                processed["X_train"].shape[1]
                ==
                processed["X_validation"].shape[1]
                ==
                processed["X_test"].shape[1]
            ),

        "leakage_free":
            bool(
                LEAKAGE_DF.loc[
                    LEAKAGE_DF["dataset_id"]
                    == dataset_id,
                    "leakage_free"
                ].iloc[0]
            ),

        "stratification_valid":
            bool(
                STRATIFICATION_DF.loc[
                    STRATIFICATION_DF["dataset_id"]
                    == dataset_id,
                    "stratification_valid"
                ].all()
            ),

        "preprocessor_saved":
            (
                PIPELINE_DIR
                / f"{dataset_id}_preprocessor.joblib"
            ).exists()
    }


    checks["all_checks_passed"] = all(
        checks.values()
    )


    FINAL_VALIDATION_RECORDS.append(
        {
            "dataset_id":
                dataset_id,

            **checks
        }
    )


FINAL_VALIDATION_DF = pd.DataFrame(
    FINAL_VALIDATION_RECORDS
)

display(
    FINAL_VALIDATION_DF
)


if not FINAL_VALIDATION_DF[
    "all_checks_passed"
].all():

    failed = (
        FINAL_VALIDATION_DF.loc[
            ~FINAL_VALIDATION_DF[
                "all_checks_passed"
            ],
            "dataset_id"
        ].tolist()
    )

    raise RuntimeError(
        "NOTEBOOK 02 VALIDATION FAILED.\n"
        f"Failed datasets: {failed}\n"
        "Do not continue to Notebook 03."
    )


FINAL_VALIDATION_PATH = (
    NOTEBOOK02_RESULTS_DIR
    / "notebook_02_final_validation.csv"
)

FINAL_VALIDATION_DF.to_csv(
    FINAL_VALIDATION_PATH,
    index=False
)


print()
print("=" * 100)
print("AIR-LLM — NOTEBOOK 02 COMPLETE")
print("=" * 100)
print()
print("All datasets passed preprocessing validation.")
print("Train/validation/test splits are ready.")
print("Preprocessing was fitted on training data only.")
print("No preprocessing leakage detected.")
print()
print(
    "READY FOR NOTEBOOK 03 — MISSINGNESS GENERATION"
)
print("=" * 100)

,dataset_id,train_exists,validation_exists,test_exists,target_exists,feature_dimensions_match,leakage_free,stratification_valid,preprocessor_saved,all_checks_passed
0,adult_income,True,True,True,True,True,True,True,True,True
1,bank_marketing,True,True,True,True,True,True,True,True,True
2,diabetes_130us,True,True,True,True,True,True,True,True,True



AIR-LLM — NOTEBOOK 02 COMPLETE

All datasets passed preprocessing validation.
Train/validation/test splits are ready.
Preprocessing was fitted on training data only.
No preprocessing leakage detected.

READY FOR NOTEBOOK 03 — MISSINGNESS GENERATION
